In [ ]:
from pathlib import Path

from openicu_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
DATASET = "mimic-iv"
OPENICU_CONCEPT_VERSION = "1.0.0"

# Shared output directory for this project.
# The R/RICU scripts use the same directory by default.
OUTPUT_ROOT = Path.home() / "output" / "openicu_yaib"

# Optional: override these paths if your local layout differs from the defaults.
# You can also set the corresponding environment variables:
#   OPENICU_YAIB_CONCEPT_ROOT
#   OPENICU_YAIB_ICUSTAYS_CSV
#   OPENICU_YAIB_RICU_CONCEPT_DICT
CONCEPT_ROOT = None
ICUSTAYS_CSV = None
RICU_CONCEPT_DICT = None

# ---------------------------------------------------------------------------
# 1) Export all available ICU hours for downstream ML/training.
# ---------------------------------------------------------------------------
all_hours = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=None,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote all-hours OpenICU YAIB-wide parquet to: {all_hours.output_path}")
all_hours.summary


In [ ]:
from openicu_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# 2) Export the first week only for R/RICU validation.
#
# This follows the RICU/YAIB convention used in the archive comparison:
# time = 0, 1, ..., 7 * 24, i.e. max time is inclusive.
# ---------------------------------------------------------------------------
MAX_HOURS = 7 * 24

one_week = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote one-week OpenICU YAIB-wide parquet to: {one_week.output_path}")
one_week.summary


## Optional R/RICU reference export

Run the R scripts before executing the comparison cell if you want to compare the OpenICU YAIB-wide parquet against R/RICU.

```bash
RICU_OUT_DIR="$HOME/output/openicu_yaib" Rscript scripts/export_ricu_dynamic_vars.R
RICU_OUT_DIR="$HOME/output/openicu_yaib" Rscript scripts/export_ricu_stay_windows.R
```

The comparison cell expects these files in `OUTPUT_ROOT`:

```text
ricu_dynamic_vars_miiv.parquet
ricu_stay_windows_miiv.parquet
```

The comparison uses the same window logic as the working archive notebook: RICU stay windows are converted to integer hours, capped to `MAX_HOURS`, and RICU dynamic rows are filtered to `start <= time <= end`.

`all_hours.output_path` is intended for downstream ML/training. `one_week.output_path` is only the bounded validation export.


In [ ]:
from openicu_yaib import (
    compare_openicu_wide_to_ricu_for_dataset,
    display_comparison_overview,
)

# Compare the one-week OpenICU parquet against R/RICU.
# Expected valid mismatch pattern for the known issue: 14 window/stay differences.
comparison = compare_openicu_wide_to_ricu_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    openicu_wide_path=one_week.output_path,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)

print(f"Wrote comparison reports to: {comparison.reports_dir}")

# The overview keeps all previous reports and additionally includes:
# - reproduction_accuracy: aggregate stay/row reproduction metrics
# - non_identical_common_stays_head: compact sample of common stays whose
#   keys or values differ (the complete report is written as CSV)
overview = display_comparison_overview(comparison)
overview


### Reproduction accuracy reports

The comparison now also writes:

- `reproduction_accuracy.csv`: aggregate stay-count, row-count, and exact-content metrics.
- `per_stay_reproduction.csv`: one row per stay with OpenICU/RICU row counts, the signed per-stay row-count error, key overlap, value mismatches, and `content_identical`.

The requested signed metrics use RICU as the denominator:

$$E_{stays}=\frac{n_{stays,OpenICU}-n_{stays,RICU}}{n_{stays,RICU}}$$

$$E_i=\frac{n_{rows,i,OpenICU}-n_{rows,i,RICU}}{n_{rows,i,RICU}}$$

The summary contains both the mean of $E_i$ across common stays and a summed common-stay form. `total_row_count_error` applies the same formula to all rows and therefore includes stays present on only one side. Absolute variants are included because positive and negative errors can otherwise cancel each other.
